Built a manufacturing analytics dashboard to monitor humanoid robot production, tracking assembly performance, failure rates, and process conditions across multiple production stages. Analyzed sensor data to identify key drivers of defects and operational inefficiencies.

## Step: Load the dataset

In [1]:
import pandas as pd

df = pd.read_csv("ai4i2020.csv")
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


We start by loading the raw manufacturing dataset to understand its structure and available features.

## Step: Add assembly stages

In [2]:
stage_map = {
    'L': 'Mechanical Assembly',
    'M': 'Electrical Wiring',
    'H': 'Final QA'
}

df['assembly_stage'] = df['Type'].map(stage_map)

Real manufacturing systems are broken into stages (assembly, wiring, QA). The dataset lacks this structure.

## Step: Add timestamps

The dataset does not include time, but time is essential for:

- throughput
- trends
- cycle time
- downtime

In [3]:
import pandas as pd

df['timestamp'] = pd.date_range(
    start='2024-01-01 08:00:00',
    periods=len(df),
    freq='2min'
)

## Step: Initial failure analysis

To understand the main causes of production failures.

In [4]:
df['station_id'] = df['Type'] + "_" + (df['UDI'] % 3).astype(str)

In [5]:
df['Machine failure'].value_counts()
df[['TWF','HDF','PWF','OSF','RNF']].sum()

TWF     46
HDF    115
PWF     95
OSF     98
RNF     19
dtype: int64

In [6]:
df.groupby('Type')['Machine failure'].mean()

Type
H    0.020937
L    0.039167
M    0.027694
Name: Machine failure, dtype: float64

In [7]:
df.groupby('Type')[['TWF','HDF','PWF','OSF','RNF']].sum()

,TWF,HDF,PWF,OSF,RNF
Type,,,,,
H,7,8,5,2,4
L,25,76,59,87,13
M,14,31,31,9,2


In [8]:
df[['Air temperature [K]', 'Process temperature [K]', 'Torque [Nm]', 'Tool wear [min]']].corr()

,Air temperature [K],Process temperature [K],Torque [Nm],Tool wear [min]
Air temperature [K],1.000000,0.876107,-0.013778,0.013853
Process temperature [K],0.876107,1.000000,-0.014061,0.013488
Torque [Nm],-0.013778,-0.014061,1.000000,-0.003093
Tool wear [min],0.013853,0.013488,-0.003093,1.000000


In [9]:
df.groupby('Type')['Machine failure'].mean()

Type
H    0.020937
L    0.039167
M    0.027694
Name: Machine failure, dtype: float64

## Step: Create failure_events table (unpivot)

Failure types are spread across multiple columns (TWF, HDF, etc.), which is inefficient for analysis.

In [10]:
failure_cols = ['TWF','HDF','PWF','OSF','RNF']

failure_events = df[df['Machine failure'] == 1].copy()

failure_events = failure_events.melt(
    id_vars=['UDI','timestamp','Product ID','Type','assembly_stage'],
    value_vars=failure_cols,
    var_name='failure_type',
    value_name='flag'
)

failure_events = failure_events[failure_events['flag'] == 1]

In [11]:
failure_events["Product ID"].unique()

<ArrowStringArray>
['L47257', 'H30501', 'L48689', 'H31096', 'L48943', 'M16856', 'M17026',
 'M17104', 'M17531', 'H32278',
 ...
 'L56594', 'L56673', 'L56833', 'L56834', 'L56839', 'L56843', 'L56844',
 'L56847', 'L57002', 'L57010']
Length: 330, dtype: str

## Step: Create aggregated metrics table

Dashboards should not compute heavy aggregations in real-time.

In [12]:
metrics = df.groupby(
    ['timestamp','Type','assembly_stage']
).agg(
    total_units=('UDI','count'),
    udi_list=('UDI', lambda x: list(x)),
    failures=('Machine failure','sum'),
    avg_torque=('Torque [Nm]','mean'),
    avg_temp=('Process temperature [K]','mean')
).reset_index()

metrics['defect_rate'] = metrics['failures'] / metrics['total_units']

## Step: Simulate downtime

Real systems track downtime, but the dataset does not include it.

Measuring production loss
Identifying operational inefficiencies

In [13]:
df['downtime_minutes'] = df['Machine failure'] * 5

In [14]:
downtime = df.groupby('Type')['downtime_minutes'].sum()

## Step: Aggregate by time (hourly)

Raw timestamps are too granular for meaningful analysis.


- Trend analysis
- Clean time-series charts
- Realistic reporting intervals

In [15]:
df['hour'] = df['timestamp'].dt.floor('h')

## Step: Compute defect rate

Absolute failure counts are misleading without context.
- Fair comparison across robot types
- Identifying systemic issues

In [16]:
metrics = df.groupby(
    ['hour','Type','assembly_stage']
).agg(
    total_units=('UDI','count'),
    udi_list=('UDI', lambda x: list(x)),
    failures=('Machine failure','sum'),
    avg_torque=('Torque [Nm]','mean'),
    avg_temp=('Process temperature [K]','mean'),
    total_downtime=('downtime_minutes','sum')
).reset_index()

metrics['defect_rate'] = metrics['failures'] / metrics['total_units']

## Step: Compute cycle time

Cycle time is a key manufacturing KPI that reflects production efficiency.

- Detecting slow stations
- Identifying bottlenecks

In [17]:
df = df.sort_values(['station_id','timestamp'])

df['prev_time'] = df.groupby('station_id')['timestamp'].shift(1)

df['cycle_time_min'] = (
    (df['timestamp'] - df['prev_time']).dt.total_seconds() / 60
)

In [18]:
cycle_time = df.groupby(
    ['hour','station_id']
)['cycle_time_min'].mean().reset_index()

## Step: Define KPIs

What we defined:

- Throughput
- Defect rate
- Downtime
- Cycle time
- Failure breakdown

Why:
- Dashboards should focus on actionable metrics, not raw data.

What this enables:

- Clear decision-making
- Alignment with manufacturing goals

In [19]:
df.to_csv("production_events.csv", index=False)
metrics.to_csv("production_metrics.csv", index=False)
failure_events.to_csv("failure_events.csv", index=False)

In [20]:
metrics.head()

,hour,Type,assembly_stage,total_units,udi_list,failures,avg_torque,avg_temp,total_downtime,defect_rate
0,2024-01-01 08:00:00,H,Final QA,5,"[11, 12, 19, 21, 28]",0,38.000000,309.180000,0,0.000000
1,2024-01-01 08:00:00,L,Mechanical Assembly,14,"[2, 3, 4, 5, 7, 8, 15, 16, 22, 24, 26, 27, 29,...",0,38.064286,309.042857,0,0.000000
2,2024-01-01 08:00:00,M,Electrical Wiring,11,"[1, 6, 9, 10, 13, 14, 17, 18, 20, 23, 25]",0,37.736364,309.054545,0,0.000000
3,2024-01-01 09:00:00,H,Final QA,4,"[39, 44, 49, 53]",0,49.225000,309.150000,0,0.000000
4,2024-01-01 09:00:00,L,Mechanical Assembly,17,"[32, 33, 34, 38, 40, 41, 42, 46, 48, 51, 52, 5...",1,40.711765,309.135294,5,0.058824


In [24]:
metrics.info()

<class 'pandas.DataFrame'>
RangeIndex: 991 entries, 0 to 990
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   hour            991 non-null    datetime64[us]
 1   Type            991 non-null    str           
 2   assembly_stage  991 non-null    str           
 3   total_units     991 non-null    int64         
 4   udi_list        991 non-null    object        
 5   failures        991 non-null    int64         
 6   avg_torque      991 non-null    float64       
 7   avg_temp        991 non-null    float64       
 8   total_downtime  991 non-null    int64         
 9   defect_rate     991 non-null    float64       
dtypes: datetime64[us](1), float64(3), int64(3), object(1), str(2)
memory usage: 93.0+ KB


In [22]:
df["timestamp"].dt.to_period('M').value_counts()


timestamp
2024-01    10000
Freq: M, Name: count, dtype: int64

In [28]:
metrics['total_units'].value_counts().sum()

np.int64(991)